# 01b · Build the Mastodon dataset

Preprocesses Mastodon data collected for two topics (Russia–Ukraine and US Elections
2024). Both topics share one identifier space for conversations, toots, authors and
instances.

Expected input layout, one sub-folder per Mastodon instance:

```
data/mastodon/raw/<topic>/<instance>/posts.json
data/mastodon/raw/<topic>/<instance>/replies.json
```

Steps:

1. Collect the URIs needed for the mappings from both topics.
2. Create deterministic ID mappings (sorted URIs, 1-based IDs).
3. Write one toot-object file per topic (root toots and replies).
4. Combine the topics into `mastodon_toots_object.csv`.
5. Write the reply edges to `replies.csv`.

In [ ]:
import csv
import json
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

## Configuration

In [ ]:
DATA_DIR = Path("../../data/mastodon")

TOPICS = {
    "Russia-Ukraine": DATA_DIR / "raw" / "Russia-Ukraine",
    "US-Elections-2024": DATA_DIR / "raw" / "US-Elections-2024",
}

MAPPING_FILES = {
    "instance": DATA_DIR / "instances.csv",
    "author": DATA_DIR / "authors.csv",
    "toot": DATA_DIR / "toots.csv",
    "conversation": DATA_DIR / "conversations.csv",
}

TOPIC_OUTPUTS = {
    "Russia-Ukraine": DATA_DIR / "mastodon_toots_object_rus_ukr.csv",
    "US-Elections-2024": DATA_DIR / "mastodon_toots_object_us-elections.csv",
}

ALL_TOOTS_OUTPUT = DATA_DIR / "mastodon_toots_object.csv"
REPLIES_OUTPUT = DATA_DIR / "replies.csv"

DATA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Collect identifiers for the shared mappings

In `posts.json`, only statuses with `in_reply_to_id = None` are conversation roots;
the root toot URI is also the conversation URI. The instance where a toot was
**found** is the folder the JSON file is in; the instance it **belongs** to is the
hostname of the toot URI. Replies are read from `replies.json`.

In [ ]:
def read_json(path):
    with path.open("r", encoding="utf-8-sig") as f:
        return json.load(f)


def collect_topic_mapping_values(data_folder, conversations, instances, toots, authors):
    posts_files = sorted(data_folder.rglob("posts.json"))
    replies_files = sorted(data_folder.rglob("replies.json"))
    total_posts = non_root_posts = 0

    for posts_file in tqdm(posts_files, desc=f"posts: {data_folder.name}"):
        instance_found = posts_file.parent.name
        try:
            posts = read_json(posts_file)
        except Exception as e:
            print(f"Error reading {posts_file}: {e}")
            continue

        total_posts += len(posts)
        for data in posts:
            if data.get("in_reply_to_id") is not None:
                non_root_posts += 1
                continue

            toot_uri = data.get("uri")
            author_uri = (data.get("account") or {}).get("uri")
            if not toot_uri or not author_uri:
                continue

            conversations.append(toot_uri)
            instances.append(instance_found)
            instance_belong = urlparse(toot_uri).hostname
            if instance_belong:
                instances.append(instance_belong)
            toots.append(toot_uri)
            authors.append(author_uri)

    for replies_file in tqdm(replies_files, desc=f"replies: {data_folder.name}"):
        try:
            replies = read_json(replies_file)
        except Exception as e:
            print(f"Error reading {replies_file}: {e}")
            continue

        # {local root status id: [reply status, ...]}
        for toot_replies in replies.values():
            for reply in toot_replies:
                toot_uri = reply.get("uri")
                author_uri = (reply.get("account") or {}).get("uri")
                if not toot_uri or not author_uri:
                    continue

                instance_belong = urlparse(toot_uri).hostname
                if instance_belong:
                    instances.append(instance_belong)
                toots.append(toot_uri)
                authors.append(author_uri)

    print(
        f"{data_folder.name}: {len(posts_files)} posts.json files, {total_posts:,} posts "
        f"({non_root_posts:,} non-root posts ignored)"
    )


conversations, instances, toots, authors = [], [], [], []
for data_folder in TOPICS.values():
    collect_topic_mapping_values(data_folder, conversations, instances, toots, authors)

## 2. Deterministic mappings

Duplicates are removed and URIs sorted before assigning IDs, so the same input
always produces the same IDs.

In [ ]:
def make_mapping(values, id_col, uri_col):
    unique = sorted(set(values))
    return pd.DataFrame({id_col: range(1, len(unique) + 1), uri_col: unique})


conversation_mapping = make_mapping(conversations, "conversation_id", "conversation_uri")
toot_mapping = make_mapping(toots, "toot_id", "toot_uri")
instance_mapping = make_mapping(instances, "instance_id", "instance_uri")
author_mapping = make_mapping(authors, "author_id", "author_uri")

conversation_mapping.to_csv(MAPPING_FILES["conversation"], index=False)
toot_mapping.to_csv(MAPPING_FILES["toot"], index=False)
instance_mapping.to_csv(MAPPING_FILES["instance"], index=False)
author_mapping.to_csv(MAPPING_FILES["author"], index=False)

print(f"Conversations: {len(conversation_mapping):,}")
print(f"Toots:         {len(toot_mapping):,}")
print(f"Instances:     {len(instance_mapping):,}")
print(f"Authors:       {len(author_mapping):,}")

## 3. Toot-object file per topic

| Column | Description |
|---|---|
| `created_at` | Creation time of the status |
| `text` | Plain text of the HTML `content` |
| `author_id` | Author |
| `followers_count`, `following_count`, `statuses_count` | Author account metadata |
| `instance_toot_belong_id` | Instance the status originates from |
| `conversation_id` | conversation id the toot belong to |
| `toot_id` | Toot/Status id |
| `instance_toot_found_id` | Instance the status was collected from |
| `replies_count`, `reblogs_count`, `favourites_count` | Status engagement counts |
| `language` | Language reported by Mastodon |
| `root_toot` | `True` for conversation roots, `False` for replies |

Keys in `replies.json` are local Mastodon status IDs, so a lookup from
`(instance, local root ID)` to the root URI is built first.

In [ ]:
COLUMNS = [
    "created_at",
    "text",
    "author_id",
    "followers_count",
    "following_count",
    "statuses_count",
    "instance_toot_belong_id",
    "conversation_id",
    "toot_id",
    "instance_toot_found_id",
    "replies_count",
    "reblogs_count",
    "favourites_count",
    "language",
    "root_toot",
]


def load_mapping(csv_path, id_col, uri_col):
    df = pd.read_csv(csv_path, dtype=str)
    return dict(zip(df[uri_col], df[id_col].astype(int)))


def clean_text(content):
    if not content:
        return ""
    return BeautifulSoup(content, "html.parser").get_text(" ", strip=True)


instance_map = load_mapping(MAPPING_FILES["instance"], "instance_id", "instance_uri")
author_map = load_mapping(MAPPING_FILES["author"], "author_id", "author_uri")
toot_map = load_mapping(MAPPING_FILES["toot"], "toot_id", "toot_uri")
conversation_map = load_mapping(MAPPING_FILES["conversation"], "conversation_id", "conversation_uri")

In [ ]:
def build_root_lookup(data_folder):
    """(instance found, local root status id) -> root toot URI."""
    root_lookup = {}
    for posts_file in sorted(data_folder.rglob("posts.json")):
        try:
            posts = read_json(posts_file)
        except Exception as e:
            print(f"Error reading {posts_file}: {e}")
            continue
        for post in posts:
            if post.get("in_reply_to_id") is not None:
                continue
            if post.get("id") is None or not post.get("uri"):
                continue
            root_lookup[(posts_file.parent.name, str(post["id"]))] = post["uri"]
    return root_lookup


def create_toot_row(data, instance_found_uri, conversation_uri, root_toot, counters):
    toot_uri = data.get("uri")
    if not toot_uri:
        return None
    toot_id = toot_map.get(toot_uri)
    if toot_id is None:
        counters["missing_toot_mapping"] += 1
        return None

    account = data.get("account") or {}
    author_uri = account.get("uri")
    if not author_uri:
        return None
    author_id = author_map.get(author_uri)
    if author_id is None:
        counters["missing_author_mapping"] += 1
        return None

    instance_belong_id = instance_map.get(urlparse(toot_uri).hostname)
    instance_found_id = instance_map.get(instance_found_uri)
    if instance_belong_id is None or instance_found_id is None:
        counters["missing_instance_mapping"] += 1
        return None

    conversation_id = conversation_map.get(conversation_uri)
    if conversation_id is None:
        counters["missing_conversation_mapping"] += 1
        return None

    return {
        "created_at": data.get("created_at"),
        "text": clean_text(data.get("content")),
        "author_id": author_id,
        "followers_count": account.get("followers_count", 0) or 0,
        "following_count": account.get("following_count", 0) or 0,
        "statuses_count": account.get("statuses_count", 0) or 0,
        "instance_toot_belong_id": instance_belong_id,
        "conversation_id": conversation_id,
        "toot_id": toot_id,
        "instance_toot_found_id": instance_found_id,
        "replies_count": data.get("replies_count", 0) or 0,
        "reblogs_count": data.get("reblogs_count", 0) or 0,
        "favourites_count": data.get("favourites_count", 0) or 0,
        "language": data.get("language"),
        "root_toot": root_toot,
    }

In [ ]:
def write_topic_toot_object(data_folder, output_csv):
    root_lookup = build_root_lookup(data_folder)
    counters = dict.fromkeys(
        ["root_rows", "reply_rows", "missing_toot_mapping", "missing_author_mapping",
         "missing_instance_mapping", "missing_conversation_mapping"],
        0,
    )
    failed_files = []

    with output_csv.open("w", newline="", encoding="utf-8") as output_file:
        writer = csv.DictWriter(output_file, fieldnames=COLUMNS, quoting=csv.QUOTE_ALL)
        writer.writeheader()

        for posts_file in tqdm(sorted(data_folder.rglob("posts.json")), desc=f"roots: {data_folder.name}"):
            try:
                posts = read_json(posts_file)
            except Exception as e:
                failed_files.append((str(posts_file), "posts", repr(e)))
                continue
            for data in posts:
                if data.get("in_reply_to_id") is not None or not data.get("uri"):
                    continue
                row = create_toot_row(data, posts_file.parent.name, data["uri"], True, counters)
                if row is not None:
                    writer.writerow(row)
                    counters["root_rows"] += 1

        for replies_file in tqdm(sorted(data_folder.rglob("replies.json")), desc=f"replies: {data_folder.name}"):
            instance_found_uri = replies_file.parent.name
            try:
                replies = read_json(replies_file)
            except Exception as e:
                failed_files.append((str(replies_file), "replies", repr(e)))
                continue
            for root_status_id, toot_replies in replies.items():
                conversation_uri = root_lookup.get((instance_found_uri, str(root_status_id)))
                if conversation_uri is None:
                    counters["missing_conversation_mapping"] += 1
                    continue
                for data in toot_replies:
                    row = create_toot_row(data, instance_found_uri, conversation_uri, False, counters)
                    if row is not None:
                        writer.writerow(row)
                        counters["reply_rows"] += 1

    print(data_folder.name, counters, f"failed_files={len(failed_files)}")
    return counters, failed_files


topic_results = {
    topic: write_topic_toot_object(folder, TOPIC_OUTPUTS[topic])
    for topic, folder in TOPICS.items()
}

## 4. Combine the topics

Duplicates are removed on `(instance_toot_found_id, conversation_id, toot_id)`: the
same toot observed on different instances is kept, repeated copies within the same
instance and conversation are dropped.

In [ ]:
all_toots = (
    pd.concat([pd.read_csv(path) for path in TOPIC_OUTPUTS.values()], ignore_index=True)
    .drop_duplicates(subset=["instance_toot_found_id", "conversation_id", "toot_id"], keep="first")
    .reset_index(drop=True)
)
all_toots.to_csv(ALL_TOOTS_OUTPUT, index=False)
print(f"Combined rows: {len(all_toots):,}")

## 5. Reply edges

Each row is a direct reply relationship `toot_id_child → toot_id_parent`, in the same
format as the Pleroma `replies.csv`. Mastodon stores the parent as a local status ID
(`in_reply_to_id`), so a lookup from `(instance, local status ID)` to the canonical
toot URI is built first.

In [ ]:
local_id_to_uri = {}
root_lookup = {}

for data_folder in TOPICS.values():
    for posts_file in sorted(data_folder.rglob("posts.json")):
        instance_found_uri = posts_file.parent.name
        try:
            posts = read_json(posts_file)
        except Exception as e:
            print(f"Error reading {posts_file}: {e}")
            continue
        for post in posts:
            status_id, toot_uri = post.get("id"), post.get("uri")
            if status_id is None or not toot_uri:
                continue
            local_id_to_uri[(instance_found_uri, str(status_id))] = toot_uri
            if post.get("in_reply_to_id") is None:
                root_lookup[(instance_found_uri, str(status_id))] = toot_uri

    for replies_file in sorted(data_folder.rglob("replies.json")):
        instance_found_uri = replies_file.parent.name
        try:
            replies = read_json(replies_file)
        except Exception as e:
            print(f"Error reading {replies_file}: {e}")
            continue
        for toot_replies in replies.values():
            for reply in toot_replies:
                status_id, toot_uri = reply.get("id"), reply.get("uri")
                if status_id is not None and toot_uri:
                    local_id_to_uri[(instance_found_uri, str(status_id))] = toot_uri

print(f"Local status lookup entries: {len(local_id_to_uri):,}")
print(f"Root lookup entries:         {len(root_lookup):,}")

In [ ]:
reply_id = 1
counts = dict.fromkeys(
    ["missing_instance", "missing_root", "missing_conversation", "missing_child",
     "missing_parent_uri", "missing_parent"],
    0,
)
failed_files = []

with REPLIES_OUTPUT.open("w", newline="", encoding="utf-8") as output_file:
    writer = csv.writer(output_file)
    writer.writerow(["instance_toot_found_id", "conversation_id", "reply_id", "toot_id_child", "toot_id_parent"])

    for data_folder in TOPICS.values():
        for replies_file in tqdm(sorted(data_folder.rglob("replies.json")), desc=f"edges: {data_folder.name}"):
            instance_found_uri = replies_file.parent.name
            instance_id = instance_map.get(instance_found_uri)
            if instance_id is None:
                counts["missing_instance"] += 1
                continue

            try:
                replies = read_json(replies_file)
            except Exception as e:
                failed_files.append((str(replies_file), repr(e)))
                continue

            for root_status_id, toot_replies in replies.items():
                conversation_uri = root_lookup.get((instance_found_uri, str(root_status_id)))
                if conversation_uri is None:
                    counts["missing_root"] += 1
                    continue
                conversation_id = conversation_map.get(conversation_uri)
                if conversation_id is None:
                    counts["missing_conversation"] += 1
                    continue

                for reply in toot_replies:
                    child_uri = reply.get("uri")
                    if not child_uri:
                        continue
                    child_id = toot_map.get(child_uri)
                    if child_id is None:
                        counts["missing_child"] += 1
                        continue

                    parent_local_id = reply.get("in_reply_to_id")
                    if parent_local_id is None:
                        continue
                    parent_uri = local_id_to_uri.get((instance_found_uri, str(parent_local_id)))
                    if parent_uri is None:
                        counts["missing_parent_uri"] += 1
                        continue
                    parent_id = toot_map.get(parent_uri)
                    if parent_id is None:
                        counts["missing_parent"] += 1
                        continue

                    writer.writerow([instance_id, conversation_id, reply_id, child_id, parent_id])
                    reply_id += 1

print(f"Reply edges written: {reply_id - 1:,}")
for key, value in counts.items():
    print(f"{key:22s} {value:,}")
print(f"Failed JSON files:   {len(failed_files):,}")